# Vector Database R&D

Objective:
Understand and compare different vector databases for Insurance RAG.

Databases Evaluated:
- FAISS
- ChromaDB
- Qdrant

Goal:
Identify the most suitable vector database for local insurance document retrieval.

In [1]:
! pip install faiss-cpu chromadb sentence-transformers

   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
    --------------------------------------- 0.3/16.1 MB ? eta -:--:--
   - -------------------------------------- 0.8/16.1 MB 3.1 MB/s eta 0:00:05
   ---- ----------------------------------- 1.8/16.1 MB 4.0 MB/s eta 0:00:04
   ----- ---------------------------------- 2.4/16.1 MB 3.9 MB/s eta 0:00:04
   ------- -------------------------------- 2.9/16.1 MB 3.8 MB/s eta 0:00:04
   ------- -------------------------------- 2.9/16.1 MB 3.8 MB/s eta 0:00:04
   -------- ------------------------------- 3.4/16.1 MB 2.6 MB/s eta 0:00:05
   ---------- ----------------------------- 4.2/16.1 MB 2.8 MB/s eta 0:00:05
   ----------- ---------------------------- 4.7/16.1 MB 2.7 MB/s eta 0:00:05
   ------------- -------------------------- 5.2/16.1 MB 2.7 MB/s eta 0:00:05
   -------------- ------------------------- 5.8/16.1 MB 2.7 MB/s eta 0:00:04
   ---------------- ----------------------- 6.8/16.1 MB 2.9 MB/s eta 0:00:04
   ----------


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import faiss
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

c:\Users\Vikram\insurance_rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
chunks = [
    "Pre-existing diseases are excluded unless declared.",
    "Premium can be paid monthly or yearly.",
    "Hospitalization expenses including room rent are covered.",
    "Cosmetic surgery is not covered under this policy.",
    "Ambulance charges are covered under policy benefits."
]

query = "What are the exclusions in this policy?"

In [4]:
model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

chunk_embeddings = model.encode(chunks)

query_embedding = model.encode([query])

print("Chunk Embeddings Shape:", chunk_embeddings.shape)
print("Query Embedding Shape:", query_embedding.shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4890.84it/s]


Chunk Embeddings Shape: (5, 384)
Query Embedding Shape: (1, 384)


In [5]:
import faiss
import numpy as np

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(chunk_embeddings).astype("float32")
)

distances, indices = index.search(
    np.array(query_embedding).astype("float32"),
    k=3
)

In [6]:
print("Top Results:\n")

for idx in indices[0]:
    print(chunks[idx])
    print("-" * 50)

Top Results:

Pre-existing diseases are excluded unless declared.
--------------------------------------------------
Cosmetic surgery is not covered under this policy.
--------------------------------------------------
Ambulance charges are covered under policy benefits.
--------------------------------------------------


# FAISS Retrieval Observation

Query:
"What are the exclusions in this policy?"

Top Results:
1. Pre-existing diseases are excluded unless declared.
2. Cosmetic surgery is not covered under this policy.
3. Ambulance charges are covered under policy benefits.

Observation:
FAISS successfully retrieved exclusion-related chunks in the top results.

The third result was less relevant because `k=3` forces the system to return three nearest chunks even if only two are strongly relevant.

Conclusion:
FAISS works well for fast local similarity search, but retrieval quality also depends on embedding model quality and the selected value of `k`.

In [7]:
! pip install chromadb langchain-chroma


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain.embeddings.base import Embeddings

In [9]:
class SentenceTransformerEmbeddings(Embeddings):

    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()

    def embed_query(self, text):
        return self.model.encode(text).tolist()

In [10]:
documents = [
    Document(
        page_content=chunk,
        metadata={"id": i}
    )
    for i, chunk in enumerate(chunks)
]

In [11]:
embedding_function = SentenceTransformerEmbeddings(model)

chroma_db = Chroma.from_documents(
    documents=documents,
    embedding=embedding_function,
    collection_name="insurance_rnd"
)

In [12]:
results = chroma_db.similarity_search(
    query,
    k=3
)

for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 50)

Pre-existing diseases are excluded unless declared.
{'id': 0}
--------------------------------------------------
Cosmetic surgery is not covered under this policy.
{'id': 3}
--------------------------------------------------
Ambulance charges are covered under policy benefits.
{'id': 4}
--------------------------------------------------


# ChromaDB Observation

ChromaDB retrieved the same relevant chunks as FAISS for the sample insurance query.

Observation:
- Top results were exclusion-related.
- Metadata was returned along with each document.
- Chroma provides a higher-level vector database interface compared to raw FAISS.

Advantages:
- Easy LangChain integration
- Metadata support
- Persistent storage support
- Suitable for local RAG prototypes

Limitation:
- Slightly heavier than FAISS

In [13]:
! pip install qdrant-client

   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.5 MB 1.3 MB/s eta 0:00:07
   ---- ----------------------------------- 1.0/9.5 MB 1.4 MB/s eta 0:00:07
   ------ --------------------------------- 1.6/9.5 MB 1.7 MB/s eta 0:00:05
   -------- ------------------------------- 2.1/9.5 MB 1.9 MB/s eta 0:00:04
   ------------ --------------------------- 2.9/9.5 MB 2.1 MB/s eta 0:00:04
   --------------- ------------------------ 3.7/9.5 MB 2.3 MB/s eta 0:00:03
   ----------------- ---------------------- 4.2/9.5 MB 2.4 MB/s eta 0:00:03
   ---------------------- ----------------- 5.2/9.5 MB 2.6 MB/s eta 0:00:02
   -------------------------- ------------- 6.3/9.5 MB 2.9 MB/s eta 0:00:02
   -------------------------------- ----


[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

In [17]:
client = QdrantClient(":memory:")

In [18]:
client.create_collection(
    collection_name="insurance_rnd",
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

True

In [19]:
points = []

for i, chunk in enumerate(chunks):
    points.append(
        PointStruct(
            id=i,
            vector=chunk_embeddings[i].tolist(),
            payload={"text": chunk}
        )
    )

client.upsert(
    collection_name="insurance_rnd",
    points=points
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [20]:
results = client.query_points(
    collection_name="insurance_rnd",
    query=query_embedding[0].tolist(),
    limit=3
).points

In [21]:
for result in results:
    print(result.payload["text"])
    print("-" * 50)

Pre-existing diseases are excluded unless declared.
--------------------------------------------------
Cosmetic surgery is not covered under this policy.
--------------------------------------------------
Ambulance charges are covered under policy benefits.
--------------------------------------------------


In [ ]:
# Final Vector Database Comparison

## Query Used

What are the exclusions in this policy?

## Retrieval Results

All three vector databases retrieved similar relevant chunks:

1. Pre-existing diseases are excluded unless declared.
2. Cosmetic surgery is not covered under this policy.
3. Ambulance charges are covered under policy benefits.

## Comparison Table

| Feature | FAISS | ChromaDB | Qdrant |
|---|---|---|---|
| Type | Vector search library | Local vector database | Production-grade vector database |
| Retrieval Quality | Good | Good | Good |
| Metadata Support | Limited/manual | Built-in | Built-in |
| Persistence | Manual | Built-in | Built-in |
| Setup Complexity | Easy | Easy | Medium |
| LangChain Integration | Available | Excellent | Available |
| Production Readiness | Medium | Medium | High |
| Best Use Case | Fast local similarity search | Local RAG prototype | Scalable production RAG |

## Decision

For the current Insurance RAG prototype, **ChromaDB** is selected.

## Reason

ChromaDB provides the best balance for the current project:

- Easy setup
- Good retrieval quality
- Metadata support
- Persistent storage
- Simple LangChain integration
- Suitable for local development and manager demo

## Future Recommendation

For large-scale production deployment with many users and millions of vectors, **Qdrant** should be considered.

FAISS remains useful when maximum local search speed is required and metadata/persistence needs are simple.